In [ ]:
"""
Berkeley Segmentation Dataset (BSDS500) - Logistic Regression vs SVM
======================================================================

WHAT THIS SCRIPT DOES
----------------------
BSDS500 is an *image* dataset (color images + hand-drawn boundary/segmentation
ground truth), not a plain tabular dataset. The standard way to turn it into a
classification problem is: for every pixel, predict EDGE (1) vs NON-EDGE (0)
using the ground-truth boundary maps as labels, and simple pixel/patch
features (color, local gradient, local texture) as inputs.

This script:
  1. Loads BSDS500 images + ground truth boundaries from disk
  2. Extracts per-pixel features (RGB, grayscale intensity, gradient
     magnitude, local standard deviation)
  3. Builds a balanced pixel-level train/test dataset (edge vs non-edge)
  4. Trains Logistic Regression and SVM classifiers
  5. Computes Accuracy, Precision, Recall, F1-score and ROC-AUC for both
  6. Prints a comparison table and plots ROC curves

HOW TO GET THE DATA
--------------------
1. Download "Berkeley Segmentation Dataset (BSDS500)" from Kaggle, e.g.:
   https://www.kaggle.com/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500
2. Unzip it. You should get a folder structure roughly like:

   BSDS500/
       images/
           train/*.jpg
           val/*.jpg
           test/*.jpg
       ground_truth/
           train/*.mat
           val/*.mat
           test/*.mat

   (Exact folder names vary slightly between Kaggle re-uploads — adjust
   IMAGES_DIR / GT_DIR below to match what you actually have.)
3. Set DATA_ROOT below to the path where you unzipped it.

If your ground truth is stored as .mat files (the original BSDS format),
this script reads them with scipy.io.loadmat. Some Kaggle mirrors instead
ship the ground truth as PNG boundary masks — see load_ground_truth_png()
as an alternative loader (swap it in if that's your case).

REQUIRED PACKAGES
------------------
pip install numpy scipy scikit-learn scikit-image opencv-python matplotlib --break-system-packages
"""

import os
import glob
import numpy as np
import cv2
from scipy import io as sio
from skimage.filters import sobel
from skimage.color import rgb2gray

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report
)
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# CONFIG - edit these paths to match your downloaded dataset
# ----------------------------------------------------------------------
DATA_ROOT   = r"C:\Users\BIT\Downloads"                              # parent of the two folders
IMAGES_DIR  = os.path.join(DATA_ROOT, "images", "train")            # adjust "train" if needed - see note below
GT_DIR      = os.path.join(DATA_ROOT, "ground_truth", "train")      # adjust "train" if needed - see note below

# NOTE: Kaggle's BSDS500 typically has train/val/test subfolders inside
# both "images" and "ground_truth". Run this once to confirm the exact
# subfolder names before running main() - uncomment to check:
#
# for d in [os.path.join(DATA_ROOT, "images"), os.path.join(DATA_ROOT, "ground_truth")]:
#     print(d, "->", os.listdir(d))

N_IMAGES_TO_USE   = 15      # how many images to use (keeps runtime sane)
PIXELS_PER_IMAGE  = 4000    # sampled pixels per image (balanced edge/non-edge)
RANDOM_STATE      = 42


# ----------------------------------------------------------------------
# 1. GROUND TRUTH LOADING
# ----------------------------------------------------------------------
def load_ground_truth_mat(mat_path, debug=False):
    """
    BSDS500 .mat ground truth contains multiple human annotations.
    Each annotation has a 'Boundaries' field (binary edge map).
    We average across annotators to get a soft edge map, then threshold.

    NOTE: scipy.io.loadmat's exact nesting for MATLAB struct/cell arrays can
    vary depending on how the .mat was saved. This function tries a couple
    of common layouts and falls back gracefully, printing diagnostics if
    debug=True.
    """
    mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    gt = mat["groundTruth"]  # with squeeze_me=True this is usually a 1D object array of structs

    if debug:
        print(f"  [debug] groundTruth type: {type(gt)}, shape: {getattr(gt, 'shape', 'N/A')}")

    # gt is an array of MATLAB struct objects (one per human annotator)
    boundary_maps = []
    for annotator in np.atleast_1d(gt):
        b = annotator.Boundaries  # with struct_as_record=False, fields are attributes
        boundary_maps.append(np.asarray(b, dtype=np.float32))

    if debug:
        print(f"  [debug] number of annotators found: {len(boundary_maps)}")
        if boundary_maps:
            print(f"  [debug] boundary map shape: {boundary_maps[0].shape}, "
                  f"min={boundary_maps[0].min()}, max={boundary_maps[0].max()}")

    avg_boundary = np.mean(boundary_maps, axis=0)
    binary_edges = (avg_boundary > 0.3).astype(np.uint8)  # majority-vote-ish threshold

    if debug:
        print(f"  [debug] edge pixels: {binary_edges.sum()}, "
              f"non-edge pixels: {(binary_edges == 0).sum()}")

    return binary_edges


def load_ground_truth_png(png_path):
    """
    Alternative loader if your Kaggle version ships boundary maps as PNGs
    instead of .mat files. Swap this in for load_ground_truth_mat if needed.
    """
    mask = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
    return (mask > 127).astype(np.uint8)


# ----------------------------------------------------------------------
# 2. FEATURE EXTRACTION
# ----------------------------------------------------------------------
def extract_features(img_bgr):
    """
    Returns an (H, W, F) feature volume for every pixel:
      - R, G, B
      - grayscale intensity
      - Sobel gradient magnitude (edge strength)
      - local standard deviation (5x5 texture window)
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    gray = rgb2gray(img_rgb)
    grad_mag = sobel(gray)

    # local std-dev via box-filtered mean of squares (fast texture feature)
    mean = cv2.blur(gray, (5, 5))
    mean_sq = cv2.blur(gray ** 2, (5, 5))
    local_std = np.sqrt(np.maximum(mean_sq - mean ** 2, 0))

    features = np.dstack([
        img_rgb[:, :, 0], img_rgb[:, :, 1], img_rgb[:, :, 2],
        gray, grad_mag, local_std
    ])
    return features  # shape (H, W, 6)


# ----------------------------------------------------------------------
# 3. BUILD PIXEL-LEVEL DATASET (balanced sampling of edge / non-edge)
# ----------------------------------------------------------------------
def build_dataset(images_dir, gt_dir, n_images, pixels_per_image, seed, debug=False):
    rng = np.random.RandomState(seed)
    image_paths = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))[:n_images]
    print(f"Found {len(image_paths)} candidate images to process.")

    X_list, y_list = [], []

    for img_path in image_paths:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        mat_path = os.path.join(gt_dir, base_name + ".mat")

        if not os.path.exists(mat_path):
            print(f"  [skip] {base_name}: no matching .mat file at {mat_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"  [skip] {base_name}: cv2.imread failed to load image")
            continue

        try:
            edges = load_ground_truth_mat(mat_path, debug=debug)
        except Exception as e:
            print(f"  [skip] {base_name}: failed to parse ground truth ({e})")
            continue

        feats = extract_features(img)

        edge_idx = np.argwhere(edges == 1)
        non_edge_idx = np.argwhere(edges == 0)

        n_each = pixels_per_image // 2
        if len(edge_idx) == 0 or len(non_edge_idx) == 0:
            print(f"  [skip] {base_name}: edge_idx={len(edge_idx)}, non_edge_idx={len(non_edge_idx)} (one class empty)")
            continue

        print(f"  [ok]   {base_name}: edge_idx={len(edge_idx)}, non_edge_idx={len(non_edge_idx)}")

        edge_sample = edge_idx[rng.choice(len(edge_idx), min(n_each, len(edge_idx)), replace=False)]
        non_edge_sample = non_edge_idx[rng.choice(len(non_edge_idx), min(n_each, len(non_edge_idx)), replace=False)]

        for (r, c) in edge_sample:
            X_list.append(feats[r, c, :])
            y_list.append(1)
        for (r, c) in non_edge_sample:
            X_list.append(feats[r, c, :])
            y_list.append(0)

    X = np.array(X_list)
    y = np.array(y_list)
    return X, y


# ----------------------------------------------------------------------
# 4. MAIN
# ----------------------------------------------------------------------
def main():
    print("Loading BSDS500 and building pixel-level edge/non-edge dataset...")
    X, y = build_dataset(IMAGES_DIR, GT_DIR, N_IMAGES_TO_USE, PIXELS_PER_IMAGE, RANDOM_STATE, debug=True)
    print(f"Dataset shape: X={X.shape}, y={y.shape}, positive rate={y.mean():.3f}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "SVM (RBF kernel)": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    }

    results = {}
    plt.figure(figsize=(7, 6))

    for name, model in models.items():
        print(f"\nTraining {name} ...")
        model.fit(X_train_s, y_train)
        y_pred = model.predict(X_test_s)
        y_proba = model.predict_proba(X_test_s)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)

        results[name] = {
            "Accuracy": acc, "Precision": prec, "Recall": rec,
            "F1": f1, "ROC-AUC": roc_auc
        }

        print(classification_report(y_test, y_pred, target_names=["Non-Edge", "Edge"]))

        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")

    plt.plot([0, 1], [0, 1], "k--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve: Logistic Regression vs SVM (BSDS500 edge detection)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig("roc_comparison.png", dpi=150)
    print("\nSaved ROC curve plot to roc_comparison.png")

    # ---- Summary table ----
    print("\n" + "=" * 65)
    print(f"{'Metric':<12}" + "".join(f"{name:>22}" for name in results))
    print("=" * 65)
    for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
        row = f"{metric:<12}"
        for name in results:
            row += f"{results[name][metric]:>22.4f}"
        print(row)
    print("=" * 65)


if __name__ == "__main__":
    main()


Loading BSDS500 and building pixel-level edge/non-edge dataset...
Found 15 candidate images to process.
  [debug] groundTruth type: <class 'numpy.ndarray'>, shape: (6,)
  [debug] number of annotators found: 6
  [debug] boundary map shape: (321, 481), min=0.0, max=1.0
  [debug] edge pixels: 3857, non-edge pixels: 150544
  [ok]   100075: edge_idx=3857, non_edge_idx=150544
  [debug] groundTruth type: <class 'numpy.ndarray'>, shape: (5,)
  [debug] number of annotators found: 5
  [debug] boundary map shape: (481, 321), min=0.0, max=1.0
  [debug] edge pixels: 2029, non-edge pixels: 152372
  [ok]   100080: edge_idx=2029, non_edge_idx=152372
  [debug] groundTruth type: <class 'numpy.ndarray'>, shape: (5,)
  [debug] number of annotators found: 5
  [debug] boundary map shape: (321, 481), min=0.0, max=1.0
  [debug] edge pixels: 4164, non-edge pixels: 150237
  [ok]   100098: edge_idx=4164, non_edge_idx=150237
  [debug] groundTruth type: <class 'numpy.ndarray'>, shape: (6,)
  [debug] number of anno